# 📗 QLoRA 학습과 평가: 4비트 베이스에 어댑터를 학습하고 LoRA·풀 파인튜닝과 비교하기

교안 01에서는 모델을 4비트로 불러오고, 학습 한 번에 드는 GPU 메모리를 측정했습니다. 이 교안은 4비트로 불러온 베이스 모델(4비트 Base) 위에 지난 시간과 같은 LoRA 어댑터를 같은 설정으로 학습합니다. 에폭별 어댑터를 지난 시간과 같은 판사 지표로 평가해 하나를 고르고, 고른 어댑터를 QLoRA Tuned라고 부릅니다. 마지막으로 QLoRA Tuned를 지난 시간 LoRA, SFT 단원 풀 파인튜닝과 품질과 학습 시간으로 비교합니다.

<img src="images/10_lesson02_goals.png" width="1050" alt="교안 02에서 배울 네 가지: 1 QLoRA 학습, 2 에폭별 평가와 고르기, 3 LoRA, 풀 파인튜닝과 품질과 학습 시간 비교, 4 어댑터 저장과 원래 정밀도 베이스에 병합.">

**학습 목표** (그림의 번호와 같습니다)

1. **4비트 Base에 지난 시간과 같은 LoRA 어댑터를 4에폭 학습하고 학습 전·후 답을 비교합니다.**
2. **에폭별 어댑터를 지난 시간과 같은 판사 지표로 평가해 QLoRA Tuned를 고릅니다.**
3. **QLoRA Tuned를 LoRA·풀 파인튜닝과 품질과 학습 시간으로 한 테이블에서 비교합니다.**
4. **어댑터를 저장하고 원래 정밀도 베이스에 병합한 모델로 답을 얻습니다.**

## 1. 실습 준비

### 1-1. 라이브러리와 경로

교안 01과 같은 준비 셀입니다. 4비트로 불러오기에 쓰는 bitsandbytes 라이브러리는 CUDA GPU에서만 동작하므로, 이 교안은 RunPod 24GB GPU에서 실행합니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

### 1-2. 지난 시간과 같은 함수와 데이터

아래 함수와 데이터는 모두 지난 시간 코드 그대로입니다. 함수마다 하는 일만 적습니다.

- **`generate_reply`, `generate_replies`:** `generate_reply`는 질문 하나에, `generate_replies`는 질문 리스트를 32개씩 묶어 그리디 생성으로 답합니다. `system_prompt=None`이면 시스템 메시지 없이 묻습니다.
- **`load_jsonl`, `to_prompt_completion`:** `load_jsonl`은 JSONL 파일을 행 리스트로 읽고, `to_prompt_completion`은 한 행의 `messages`를 prompt와 completion으로 나눕니다.
- **질문 여섯 개(`check_rows`), `answer_check_questions`:** `check_rows`는 지난 시간 교안 01과 같은 행(학습 데이터 2, 검증 데이터 1, 범위 밖 질문 1, 일반 질문 2)입니다. `answer_check_questions`는 이 여섯 질문에 모델이 한 답을 같은 순서의 리스트로 돌려줍니다.

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 지난 시간과 같은 토크나이저(chat template 포함). HF_TOKEN 이 없다는 Warning 은 로그인 없이 내려받는다는 안내이며 오류가 아닙니다
tokenizer =AutoTokenizer.from_pretrained(MODEL_NAME)

# 지난 시간 데이터의 시스템 메시지와 글자까지 같은 문장
SYSTEM_PROMPT = "너는 노무 상담 챗봇이다. 노동법 조문에 근거해 존댓말로 간결히 답한다."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_reply(model, question, system_prompt=SYSTEM_PROMPT, max_new_tokens=256):
    """질문 하나를 학습 데이터와 같은 chat template으로 감싸 모델이 새로 만든 답만 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
        enable_thinking=False,       # 빈 생각 블록(<think></think>)을 붙여 생각 과정(thinking) 없이 바로 답합니다
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한. 답이 이보다 길면 끝까지 쓰기 전에 멈춥니다
            do_sample=False,                       # 매번 확률이 가장 높은 토큰을 골라(그리디 생성) 같은 질문에 같은 답이 나옵니다
            eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
            pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
        )

    # 입력 프롬프트 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def generate_replies(model, questions, system_prompt, max_new_tokens=256):
    """질문을 32개씩 묶어 생성하고, 질문 순서대로 답 목록을 돌려줍니다. system_prompt=None 이면 시스템 메시지 없이 묻습니다."""
    # 여러 질문을 묶으면 짧은 질문을 패딩으로 채워 길이를 맞춥니다. 답은 오른쪽 끝에 이어 쓰므로 패딩은 왼쪽에 둡니다
    tokenizer.padding_side = "left"
    answers = []
    for start in range(0, len(questions), 32):
        # generate_reply 와 같은 chat template으로 질문을 감쌉니다
        prompts = []
        for question in questions[start:start + 32]:
            messages = []
            if system_prompt is not None:
                messages.append({"role": "system", "content": system_prompt})
            messages.append({"role": "user", "content": question})
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)

        # generate_reply 와 같은 그리디 생성
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                        eos_token_id=END_OF_TURN_ID, pad_token_id=tokenizer.pad_token_id)

        # 입력 길이 뒤의 새 토큰만 글자로 바꿉니다(먼저 끝난 답 뒤의 패딩은 skip_special_tokens 로 빠집니다)
        for new_ids in output_ids[:, inputs["input_ids"].shape[1]:]:
            answers.append(tokenizer.decode(new_ids, skip_special_tokens=True))

    tokenizer.padding_side = "right"  # 학습기가 쓰는 오른쪽 패딩으로 되돌립니다
    return answers


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
from datasets import Dataset

# 학습, 검증 파일(지난 시간과 같은 노무 상담 데이터)을 prompt와 completion으로 나눕니다
train_rows = load_jsonl(data_dir / "본실습" / "labor_train.jsonl")
val_rows = load_jsonl(data_dir / "본실습" / "labor_val.jsonl")
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])

# 평가 파일: 노무 평가셋(재현, 적용), 범위 밖 질문, 일반 질문, 일반 문단
eval_rows = load_jsonl(data_dir / "본실습" / "labor_eval.jsonl")
out_of_scope_rows = load_jsonl(data_dir / "본실습" / "out_of_scope_eval.jsonl")
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")
general_texts = json.loads((data_dir / "본실습" / "general_texts.json").read_text(encoding="utf-8"))

# 학습 데이터 표본: 거절 답 행을 뺀 학습 행에서 같은 간격으로 고른 40행(지난 시간 교안 02와 같은 행)
answer_rows = []
for row in train_rows:
    if "답할 수 없" not in row["answer"]:
        answer_rows.append(row)
sample_gap = len(answer_rows) // 40
train_sample_rows = []
for row in answer_rows[::sample_gap][:40]:
    # 학습한 답(answer)을 정답 답변으로 씁니다
    train_sample_rows.append({"question": row["question"], "reference_answer": row["answer"]})
print("학습:", len(train_dataset), "| 검증:", len(val_dataset), "| 노무 평가셋:", len(eval_rows), "| 범위 밖:", len(out_of_scope_rows),
      "| 일반 질문:", len(general_rows), "| 일반 문단:", len(general_texts), "| 학습 데이터 표본:", len(train_sample_rows))

In [ ]:
# 지난 시간 데이터가 상담 범위 밖 질문에 단 거절 답(지난 시간 데이터 만들기와 같은 문장)
REFUSAL_ANSWER = "이 상담 챗봇이 다루는 노동법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 학습 데이터에서 정답 답변이 거절 답인 첫 행
for row in train_rows:
    if row["answer"] == REFUSAL_ANSWER:
        refusal_row = row
        break

# 학습 전·후에 똑같이 물을 질문 여섯 개와 그 종류(지난 시간 교안 01과 같은 행)
check_rows = [train_rows[300], train_rows[900], val_rows[39], refusal_row, general_rows[0], general_rows[12]]
check_kinds = ["학습 데이터", "학습 데이터", "검증 데이터", "범위 밖 질문", "일반 질문", "일반 질문"]


def answer_check_questions(model):
    """여섯 질문에 model이 한 답을 같은 순서의 리스트로 돌려줍니다. 일반 질문은 시스템 프롬프트 없이 묻습니다."""
    answers = []
    for row, kind in zip(check_rows, check_kinds):
        if kind == "일반 질문":
            answers.append(generate_reply(
                model,               # 답을 만들 모델
                row["question"],     # 질문 문장
                system_prompt=None,  # 일반 질문은 시스템 메시지 없이 묻습니다
            ))
        else:
            answers.append(generate_reply(
                model,            # 답을 만들 모델
                row["question"],  # 질문 문장(시스템 프롬프트는 기본값 SYSTEM_PROMPT)
            ))
    return answers

### 1-3. 지난 시간과 같은 판사와 평가 함수

QLoRA 어댑터를 지난 시간 LoRA, 풀 파인튜닝과 비교하려면 평가 방법이 같아야 합니다. 그래서 판사와 지표 함수도 지난 시간 코드 그대로 씁니다.

- **`judge_answers`, `answer_and_judge`:** Jev 판사가 모델의 답을 정답 답변(노무 질문은 근거 조문도)과 비교해 통과(1) 또는 실패(0)로 판정합니다. 범위 밖 질문은 판사를 부르지 않고, 답에 거절 키워드가 있는지로 셉니다.
- **`perplexity`, `evaluate_model`, `table_metrics`:** 모델 하나를 학습 데이터 표본, 노무 평가셋, 범위 밖 질문, 일반 질문, 일반 문단으로 평가해 지표 일곱 개를 냅니다.
- **API 키와 호출 수:** `.env`의 `TYPESAFE_API_KEY`가 필요합니다. 모델 하나에 판사를 119번(학습 데이터 표본 40 + 노무 평가셋 49 + 일반 질문 30) 부릅니다. 이 교안은 모델 5개(4비트 Base, 에폭 어댑터 4개)를 평가하므로 모두 595번 부릅니다.

In [ ]:
import os

from dotenv import load_dotenv

# 단원 폴더의 .env를 먼저 읽고, 없는 키만 상위 실습자료 폴더의 .env에서 채웁니다
load_dotenv(material_dir / ".env")
load_dotenv(material_dir.resolve().parent / ".env")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "1"  # 사용 기록을 외부로 보내지 않음(deepeval을 불러오기 전에 설정)
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig, DisplayConfig
from deepeval.metrics import JevEval
from deepeval.metrics.jev_eval import Noul
from deepeval.test_case import LLMTestCase, SingleTurnParams

# 판정 결과가 모델 버전에 따라 달라지지 않게 Jev 버전을 고정합니다
jev_model_name = os.getenv("TYPESAFE_MODEL", "jev-1.13.0")
print("Jev 모델:", jev_model_name)
print("Jev 키 읽음:", os.getenv("TYPESAFE_API_KEY") is not None)

In [ ]:
# 두 판사가 함께 쓰는 판정 문장. Jev가 문장마다 참일 확률을 돌려줍니다
BASE_STATEMENTS = [
    Noul("actual_output은 input의 질문에 직접 답한다."),
    Noul("actual_output의 사실(숫자, 기간, 조건, 이름)과 결론이 expected_output과 일치한다. 표현이 달라도 같은 정보를 전하면 일치로 본다."),
    Noul("actual_output은 input의 질문과 관계없는 지어낸 내용을 덧붙이지 않는다."),
    Noul("actual_output은 같은 문장이나 구절을 되풀이하지 않는다."),
]

# 일반 질문과 학습 데이터 표본용: 질문, 모델의 답, 정답 답변을 봅니다
general_metric = JevEval(
    name="정확성",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],  # 판정 문장에서 가리키는 필드
    questions=BASE_STATEMENTS,
    system_one_model=jev_model_name,
    strict_mode=True,  # 모든 문장의 참일 확률이 0.5 이상이어야 통과
    async_mode=True,  # Jev 호출을 비동기로 보냄(DeepEval 기본값)
)

# 노무, 전월세 질문용: 근거 조문(context)도 보고, 정답 답변에 없는 말이 조문과 어긋나지 않는지 한 문장 더 묻습니다
law_metric = JevEval(
    name="정확성(근거 조문)",
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT, SingleTurnParams.CONTEXT],
    questions=BASE_STATEMENTS + [
        Noul("actual_output이 expected_output에 없는 내용을 말할 때, 그 내용은 context(근거 조문)와 모순되지 않는다."),
    ],
    system_one_model=jev_model_name,
    strict_mode=True,
    async_mode=True,
)


def judge_answers(rows, answers):
    """판사가 답 목록을 동시에 채점해, 문항 순서대로 점수, 통과, 이유 열을 가진 테이블을 돌려줍니다."""
    # rows 는 모두 같은 종류입니다. 근거 조문(law_text)이 있으면 근거 조문 기준으로 채점합니다
    metric = general_metric
    if "law_text" in rows[0]:
        metric = law_metric

    # 문항마다 테스트 케이스 하나
    test_cases = []
    for index, row in enumerate(rows):
        context = None
        if "law_text" in row:
            context = [row["law_text"]]
        test_cases.append(LLMTestCase(
            name=str(index),                          # 결과를 문항 순서로 되돌리는 번호(동시 채점은 끝나는 순서가 섞입니다)
            input=row["question"],                    # 질문
            actual_output=answers[index],             # 채점할 모델의 답
            expected_output=row["reference_answer"],  # 정답 답변
            context=context,                          # 근거 조문 원문(없으면 None)
        ))

    # 판사를 동시에 부릅니다. 동시에 보낼 문항 수 상한은 64이고, 반 전체가 같은 키를 쓰다 429(요청 한도 초과)가 나면 줄입니다
    result = evaluate(test_cases=test_cases, metrics=[metric], async_config=AsyncConfig(max_concurrent=64),
                      display_config=DisplayConfig(print_results=False, show_indicator=False))  # 긴 기본 출력을 끕니다

    # 결과를 번호(name)로 찾아 문항 순서대로 놓습니다. 점수는 모든 문장이 참이면 1, 아니면 0이고, 이유에는 문장마다 참일 확률이 나옵니다
    judged = {}
    for test_result in result.test_results:
        judged[test_result.name] = test_result.metrics_data[0]  # 지표를 하나만 넘겼습니다
    judged_rows = []
    for index in range(len(rows)):
        metric_data = judged[str(index)]
        judged_rows.append({"점수": metric_data.score, "통과": metric_data.success, "이유": metric_data.reason})
    return pd.DataFrame(judged_rows)

In [ ]:
def answer_and_judge(model, rows, system_prompt):
    """rows의 질문에 답을 생성하고 채점해, 질문마다 한 행씩 모은 테이블을 돌려줍니다."""
    questions = []
    for row in rows:
        questions.append(row["question"])
    answers = generate_replies(model, questions, system_prompt)

    # 거절은 거절 키워드가 답에 있는지로 셉니다(거절률과 과잉 거절률에 씁니다)
    result_rows = []
    for row, answer in zip(rows, answers):
        result_rows.append({"질문": row["question"], "종류": row.get("question_type", ""), "답변": answer,
                            "거절": "답할 수 없" in answer})
    result_table = pd.DataFrame(result_rows)

    # 정답 답변이 있는 행만 판사가 채점합니다(범위 밖 질문은 정답 답변이 없습니다)
    if "reference_answer" in rows[0]:
        result_table = pd.concat([result_table, judge_answers(rows, answers)], axis=1)
    return result_table

In [ ]:
import math


def perplexity(model, texts):
    """문단 목록 전체의 perplexity: 예측한 토큰 전체의 평균 손실에 exp를 합니다."""
    total_loss = 0.0
    total_tokens = 0
    for text in texts:
        input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
        with torch.no_grad():
            loss = model(input_ids, labels=input_ids).loss.item()  # 입력을 정답으로 주면 다음 토큰 예측 손실의 평균을 돌려줍니다

        # 첫 토큰은 예측하지 않으므로 예측한 토큰 수는 길이 - 1입니다. 평균 * 개수 = 이 문단의 손실 합
        n_predicted = input_ids.shape[1] - 1
        total_loss = total_loss + loss * n_predicted
        total_tokens = total_tokens + n_predicted
    return math.exp(total_loss / total_tokens)


# 지표 이름(MLflow 에 기록할 이름)과 테이블에 보일 이름. 학습 데이터 정답률을 맨 앞에 둡니다
METRIC_NAMES = {"train_acc": "학습 데이터 정답률", "domain_recall_acc": "도메인 정답률(재현)", "domain_apply_acc": "도메인 정답률(적용)",
                "over_refusal_rate": "과잉 거절률", "refusal_rate": "거절률", "general_acc": "일반 질문 정답률", "general_ppl": "일반 perplexity"}


def evaluate_model(model):
    """모델 하나를 앞의 평가 네 가지와 perplexity 로 평가해 사전으로 돌려줍니다."""
    return {
        "train": answer_and_judge(model, train_sample_rows, SYSTEM_PROMPT),
        "domain": answer_and_judge(model, eval_rows, SYSTEM_PROMPT),
        "refusal": answer_and_judge(model, out_of_scope_rows, SYSTEM_PROMPT),
        "general": answer_and_judge(model, general_rows, None),
        "ppl": perplexity(model, general_texts),
    }


def table_metrics(tables):
    """evaluate_model 이 돌려준 사전으로 지표 일곱 개를 계산합니다. 정답률, 거절률은 True 를 1로 본 평균입니다."""
    domain_table = tables["domain"]
    return {
        "train_acc": tables["train"]["통과"].mean(),
        "domain_recall_acc": domain_table[domain_table["종류"] == "재현"]["통과"].mean(),
        "domain_apply_acc": domain_table[domain_table["종류"] == "적용"]["통과"].mean(),
        "over_refusal_rate": domain_table["거절"].mean(),
        "refusal_rate": tables["refusal"]["거절"].mean(),
        "general_acc": tables["general"]["통과"].mean(),
        "general_ppl": tables["ppl"],
    }

### 1-4. MLflow

학습 기록과 학습하는 동안의 최대 할당 메모리는 이 단원의 MLflow 실험 `labor-qlora`에 남깁니다.

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
# 처음 실행하면 기록 파일과 실험을 새로 만든다는 INFO 줄이 나옵니다. 오류가 아닙니다
mlflow.set_experiment("labor-qlora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

### 1-5. 4비트 Base와 LoRA 설정

교안 01과 같이 `bnb_config`로 4비트 Base를 불러오고, 지난 시간과 같은 `lora_config`를 만듭니다. 어댑터는 2-3에서 `SFTTrainer`가 이 4비트 Base에 바로 붙입니다. 그래서 그 전에 2-1에서 4비트 Base의 학습 전 답을 모읍니다.

In [ ]:
from transformers import BitsAndBytesConfig

# QLoRA 논문과 같은 4비트 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 선형 층 가중치를 4비트로 불러옵니다
    bnb_4bit_quant_type="nf4",              # 기본값 "fp4" 대신 NF4로 양자화합니다
    bnb_4bit_use_double_quant=True,         # 블록마다 둔 absmax도 8비트로 한 번 더 양자화합니다
    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산할 때 bf16으로 되돌립니다(기본값 float32는 느립니다)
)

In [ ]:
# 선형 층은 4비트로, 나머지 층(임베딩, 정규화 등)은 bf16으로 불러옵니다
base_4bit = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                      # 지난 시간과 같은 모델
    quantization_config=bnb_config,  # 위 4비트 설정
    dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층의 dtype
    device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

In [ ]:
from peft import LoraConfig

# 지난 시간 본실습과 같은 설정: 출력층을 뺀 모든 선형 층에 r=8 어댑터, 배율 16 / 8 = 2.0
lora_config = LoraConfig(
    r=8,                          # B, A 사이의 크기(rank)
    lora_alpha=16,                # 배율 = lora_alpha / r = 2.0
    target_modules="all-linear",  # lm_head를 뺀 모든 선형 층
    lora_dropout=0.05,            # 어댑터 입력에 거는 드롭아웃 비율
    task_type="CAUSAL_LM",        # 다음 토큰을 예측하는 생성 모델
)

## 2. QLoRA 학습

### 2-1. 학습 전 답

질문 여섯 개를 float32 Base(지난 시간 LoRA의 출발점)와 4비트 Base(QLoRA의 출발점)에 물어, 4비트로 불러오기만 해도 답이 바뀌는지 확인합니다. 4비트 Base의 학습 전 답은 여기서 모아 둡니다. 2-3에서 `SFTTrainer`가 이 모델(`base_4bit`)에 어댑터를 바로 붙이기 때문입니다. float32 Base는 답을 모은 뒤 GPU 메모리에서 지웁니다.

In [ ]:
import gc

# 질문마다 정답 답변: 노무 행은 answer, 일반 질문은 reference_answer
reference_answers = []
for row in check_rows:
    if "answer" in row:
        reference_answers.append(row["answer"])
    else:
        reference_answers.append(row["reference_answer"])

# float32 Base: 지난 시간과 같은 방법으로 불러와 답을 모으고 GPU 메모리에서 지웁니다
# causal_conv1d, flash-linear-attention 이 없다는 안내는 빠른 커널 대신 PyTorch 기본 구현으로 계산한다는 뜻이며 오류가 아닙니다
float32_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,           # 지난 시간과 같은 모델
    dtype=torch.float32,  # 지난 시간 LoRA의 베이스와 같은 float32
).to(device)
float32_answers = answer_check_questions(float32_model)
del float32_model
gc.collect()              # 더 이상 쓰지 않는 객체를 메모리에서 정리합니다
torch.cuda.empty_cache()  # PyTorch가 잡아 두었던 빈 GPU 메모리를 돌려줍니다

# 종류, 질문, 정답 답변, 두 Base의 답을 한 테이블에 둡니다
check_table = pd.DataFrame({
    "종류": check_kinds,
    "질문": [row["question"] for row in check_rows],
    "정답 답변": reference_answers,
    "float32 Base 답변": float32_answers,
    "4비트 Base 답변": answer_check_questions(base_4bit),
})
display(check_table)

**결과 읽기**
- **글자까지 같은 답:** 강사 실행에서 두 Base의 답이 글자까지 같은 질문은 여섯 가운데 0개입니다. 4비트로 불러오기만 해도 그리디 생성의 답이 모두 바뀌므로, QLoRA는 지난 시간 LoRA와 다른 출발점에서 학습합니다.
- **결론이 바뀐 답:** 학습 데이터 두 질문은 4비트 Base의 답에서 숫자가 바뀌었습니다(float32 Base -> 4비트 Base: 유산 휴가 1~2년 -> 2024년 날짜, 조정안 기한 20일 -> 제출 즉시). 두 질문 모두 두 Base의 답이 정답 답변과 다릅니다.
- **결론이 같은 답:** 두 Base의 결론이 같은 질문은 휴게시간(둘 다 1시간), 연말정산(둘 다 거절하지 않음), 서울, 광합성입니다. 다만 4비트 Base는 광합성 답에서 같은 구절을 되풀이합니다.
- **비교 기준:** 2-5의 학습 전·후 비교는 4비트 Base 답변 열을 기준으로 합니다.

### 2-2. LoRA 설정 넘기기

어댑터는 직접 붙이지 않습니다. 2-3에서 4비트 Base(`base_4bit`)와 `peft_config=lora_config`를 함께 `SFTTrainer`에 넘기면 트레이너가 어댑터를 붙입니다([TRL 문서](https://huggingface.co/docs/trl/main/en/peft_integration)가 권하는 방법). 어댑터를 붙인 모델(PeftModel)은 `trainer.model`로 꺼내 씁니다.

- **어댑터 붙이기:** 트레이너는 베이스 가중치를 고정하고 어댑터만 학습하게 둡니다. 4비트로 바뀌는 것은 베이스뿐이므로 학습 파라미터 수는 지난 시간 LoRA와 같은 5,411,328개여야 합니다.
- **어댑터 dtype:** 베이스가 4비트 모델이면 트레이너가 어댑터를 bf16으로 바꿉니다(QLoRA 논문과 같은 방식). 그래서 저장되는 어댑터 파일은 지난 시간 LoRA 어댑터(fp32)의 절반 크기입니다.

### 2-3. 학습 설정과 학습

지난 시간 LoRA와 품질을 비교하려고 지난 시간 lr 2e-4 LoRA 학습과 같은 설정(배치 4, 그래디언트 체크포인팅 끔)으로 학습하고, 폴더와 run 이름만 바꿉니다. 학습률과 갱신 횟수가 같으므로 같은 에폭의 어댑터끼리 비교할 수 있습니다.

In [ ]:
from trl import SFTConfig, SFTTrainer

# 지난 시간 lr 2e-4 LoRA 학습과 같은 값입니다. 다른 줄에만 '다름'을 적었습니다
sft_config = SFTConfig(
    output_dir=str(output_dir / "labor_qlora_lr2e-4"),  # 다름: 에폭마다 checkpoint-<갱신 횟수> 폴더가 여기에 생깁니다
    learning_rate=2e-4,                        # 지난 시간 LoRA와 같은 학습률
    num_train_epochs=4,                          # 중간에 멈추지 않고 끝까지 학습합니다
    per_device_train_batch_size=4,               # 한 번 갱신에 넣는 예시 수
    seed=0,                                      # 같은 순서로 섞어 지난 시간과 같은 배치를 만듭니다
    gradient_checkpointing=False,                # 지난 시간 LoRA와 같은 조건(체크포인팅 끔)
    max_length=1024,                             # 가장 긴 예시보다 긴 길이 상한(자르지 않음)
    eval_strategy="epoch",                       # 에폭이 끝날 때마다 검증 손실을 계산해 기록합니다
    save_strategy="epoch",                       # 에폭이 끝날 때마다 어댑터를 저장합니다
    save_only_model=True,                        # AdamW 상태 없이 어댑터만 저장합니다
    logging_steps=50,                            # 학습 손실을 기록하는 갱신 간격
    eval_on_start=True,                          # 학습 전(0에폭) 검증 손실도 기록합니다
    report_to=["mlflow"],                        # 기록을 MLflow에 남깁니다
    run_name="qlora_lr_2e-4",                     # 다름: MLflow run 이름
)

# 지난 시간과 같은 SFTTrainer 에 4비트 Base 와 LoRA 설정을 함께 넘깁니다. 어댑터는 트레이너가 붙입니다
trainer = SFTTrainer(
    model=base_4bit,             # 어댑터를 붙이지 않은 4비트 Base
    args=sft_config,             # 위 학습 설정
    train_dataset=train_dataset, # 노무 상담 학습 데이터
    eval_dataset=val_dataset,    # 에폭마다 검증 손실을 계산할 데이터
    processing_class=tokenizer,  # chat template 으로 prompt와 completion을 토큰으로 바꿉니다
    peft_config=lora_config,     # 이 설정으로 어댑터를 붙이고 어댑터만 학습합니다
)

# 트레이너가 어댑터를 붙인 모델(PeftModel). 학습 전·후 답과 학습 뒤 셀에서 이 모델을 씁니다
qlora_model = trainer.model
qlora_model.print_trainable_parameters()

**결과 읽기**
- **trainable params:** 5,411,328개로 지난 시간 LoRA와 같습니다. 4비트로 불러오기는 파라미터 수가 아니라 베이스 파라미터 하나를 저장하는 바이트 수만 바꿉니다.

In [ ]:
# 학습하는 동안의 최대 할당 메모리만 남기려고 지금까지의 최댓값 기록을 지웁니다
torch.cuda.reset_peak_memory_stats()

# 데이터 파일, 행 수, LoRA·4비트 설정을 run에 남기고 학습합니다. 갱신 횟수 = 올림(1169 / 4) * 4 = 1172
with mlflow.start_run(run_name=sft_config.run_name):
    mlflow.log_params({
        "train_file": "labor_train.jsonl",  # 학습 파일
        "train_rows": len(train_dataset),   # 학습 행 수
        "val_file": "labor_val.jsonl",      # 검증 파일
        "val_rows": len(val_dataset),       # 검증 행 수
        "lora_r": lora_config.r,            # 어댑터 rank
        "load_in_4bit": True,               # 베이스를 4비트로 불러왔습니다
        "bnb_4bit_quant_type": "nf4",       # 4비트 형식
    })
    # 토크나이저의 PAD/BOS/EOS 토큰에 맞춰 모델 설정을 고쳤다는 안내가 나옵니다. 오류가 아닙니다
    train_output = trainer.train()

    # 학습 시간(초)과 학습하는 동안의 최대 할당 메모리(GB). 메모리는 같은 run에 남깁니다
    train_seconds = train_output.metrics["train_runtime"]
    max_allocated_gb = torch.cuda.max_memory_allocated() / 1024 ** 3
    mlflow.log_metric("max_memory_allocated_gb", max_allocated_gb)
print("학습 시간(초):", round(train_seconds), "| 최대 할당 메모리(GB):", round(max_allocated_gb, 2))

**결과 읽기**
- **학습 시간과 최대 할당 메모리:** 강사 실행에서 461초, 5.33GB입니다. 메모리가 교안 01 6-2의 측정값(16.59GB)보다 작은 이유는 6-2가 배치는 같지만 시퀀스 길이 512으로 측정했기 때문입니다. 노무 예시는 평균 약 111토큰이라 활성값과 로짓이 훨씬 작습니다.

### 2-4. 학습 곡선

학습이 정상으로 진행됐는지 확인하려고 학습 손실과 검증 손실을 에폭 축에 그립니다. 지난 시간처럼 이 곡선은 학습 경과를 확인하는 기록입니다. 어댑터는 이 곡선이 아니라 3절의 판사 지표로 고릅니다.

In [ ]:
# 학습기 기록에서 학습 손실이 있는 줄과 검증 손실이 있는 줄을 나눕니다
log_table = pd.DataFrame(trainer.state.log_history)
train_log = log_table.dropna(subset=["loss"])
eval_log = log_table.dropna(subset=["eval_loss"])

# 학습 손실과 검증 손실을 에폭 축에 그립니다. 검증 손실은 학습 전(0에폭)부터 있습니다
plt.figure(figsize=(7, 4))
plt.plot(
    train_log["epoch"],        # 가로축: 에폭
    train_log["loss"],         # 세로축: 학습 손실(50번 갱신마다의 평균)
    marker="o",                # 기록마다 점을 찍습니다
    color="#2563eb",           # 파랑
    label="학습 손실(loss)",   # 범례 이름
)
plt.plot(
    eval_log["epoch"],              # 가로축: 에폭
    eval_log["eval_loss"],          # 세로축: 검증 손실
    marker="o",                     # 에폭마다 점을 찍습니다
    color="#dc2626",                # 빨강
    label="검증 손실(eval_loss)",   # 범례 이름
)
plt.title("QLoRA 학습 곡선 (lr 2e-4)")
plt.xlabel("에폭 (epoch)")
plt.ylabel("손실 (크로스 엔트로피)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**그래프 읽기**
- **검증 손실:** 강사 실행에서 학습 전 2.98, 1에폭 1.61, 2에폭 1.55(가장 낮음), 3에폭 1.59, 4에폭 1.67입니다. 4에폭 끝 학습 손실은 0.23까지 내려갑니다.
- **읽는 법:** 2에폭 뒤로는 학습 손실만 내려가고 검증 손실은 오릅니다. 2에폭 뒤로는 모델이 학습 데이터를 외우는 쪽으로 학습한다는 뜻입니다.
- **MLflow 화면:** 단원 폴더에서 `uv run mlflow server --backend-store-uri sqlite:///output/mlflow.db --port 5050`을 실행하면 실험 `labor-qlora`의 run에서 손실과 `max_memory_allocated_gb`를 볼 수 있습니다.

### 2-5. 학습 전·후 답

학습을 마친 QLoRA 모델(`qlora_model`, 4에폭 끝 상태)에 같은 질문 여섯 개를 물어, 2-1의 테이블(`check_table`)에 열로 더합니다. 답을 만들기 전에 `eval()`로 어댑터의 드롭아웃을 끕니다.

In [ ]:
# 드롭아웃을 끄고(평가 모드), 같은 여섯 질문을 QLoRA 모델에 물어 열로 더합니다
qlora_model.eval()
check_table["QLoRA 답변"] = answer_check_questions(qlora_model)
display(check_table)

**결과 읽기**
- **학습 데이터와 범위 밖 질문:** 강사 실행에서 QLoRA 답변은 정답 답변처럼 한 문장 존댓말로 바뀌었습니다. 조정안 기한(60일)과 연말정산 거절 답은 정답 답변과 글자까지 같습니다.
- **일반 질문:** QLoRA 모델은 광합성 질문에 '광합'이라고 답해, 학습 전 4비트 Base가 맞히던 질문을 틀립니다.
- **읽는 법:** 질문이 여섯 개뿐이라 경향만 확인할 수 있습니다. 쓸 어댑터는 3절에서 판사 지표로 고릅니다.

## 3. 에폭별 평가와 QLoRA Tuned 고르기

### 3-1. 4비트 Base 지표

학습에 쓴 `base_4bit`에는 트레이너가 어댑터를 붙였으므로, 학습기와 함께 지웁니다. 그리고 같은 `bnb_config`로 평가용 4비트 Base(`eval_base`)를 새로 불러옵니다(학습 때와 같은 베이스). 3-2에서 어댑터를 얹으면 `eval_base` 안의 층이 바뀌므로, 얹기 전에 4비트 Base의 지표부터 계산합니다(판사 호출 119번).

In [ ]:
# 학습기와 학습한 모델을 지웁니다. 어댑터는 에폭마다 체크포인트 폴더에 저장되어 있습니다(한 번만 실행)
del trainer, qlora_model, base_4bit
gc.collect()
torch.cuda.empty_cache()

# 평가용 4비트 Base: 학습 때와 같은 방법으로 불러옵니다
eval_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                      # 지난 시간과 같은 모델
    quantization_config=bnb_config,  # 학습 때와 같은 4비트 설정
    dtype=torch.bfloat16,            # 4비트로 바뀌지 않는 층의 dtype
    device_map={"": 0},              # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

# 4비트 Base의 채점 테이블 네 개와 perplexity, 지표 일곱 개
# 판사 호출마다 DeepEval 이 Pass Rate 요약, 'No hyperparameters logged' 경고, Confident AI 안내를 출력합니다. 오류가 아닙니다
base_tables = evaluate_model(eval_base)
model_metrics = {"4비트 Base": table_metrics(base_tables)}

### 3-2. 에폭별 어댑터 4개 얹기

지난 시간과 같은 방법으로 4비트 Base 하나에 에폭별 어댑터 4개를 얹고, 어댑터마다 이름(`ep1`~`ep4`)을 붙입니다. 첫 어댑터는 `PeftModel.from_pretrained`로, 나머지는 `load_adapter`로 얹습니다. 베이스는 다시 불러오지 않습니다.

In [ ]:
from peft import PeftModel

# 에폭마다 저장한 체크포인트 폴더의 갱신 횟수(293, 586, 879, 1172)
qlora_dir = output_dir / "labor_qlora_lr2e-4"
steps = []
for path in qlora_dir.glob("checkpoint-*"):
    steps.append(int(path.name.split("-")[1]))
steps.sort()

# 어댑터마다 이름(ep<에폭>), 에폭, 갱신 횟수, 폴더를 한 행으로 모읍니다
checkpoint_rows = []
for epoch, step in enumerate(steps, start=1):
    checkpoint_rows.append({"이름": f"ep{epoch}", "에폭": epoch, "갱신 횟수": step, "폴더": qlora_dir / f"checkpoint-{step}"})

# 첫 어댑터를 얹으며 PeftModel 을 만들고, 나머지 어댑터를 같은 모델에 이름을 붙여 더 얹습니다
qlora_model = PeftModel.from_pretrained(
    eval_base,                               # 평가용 4비트 Base
    checkpoint_rows[0]["폴더"],              # 1에폭 체크포인트 폴더
    adapter_name=checkpoint_rows[0]["이름"], # 어댑터를 켤 때 부를 이름
)
for row in checkpoint_rows[1:]:
    qlora_model.load_adapter(
        row["폴더"],               # 그 에폭의 체크포인트 폴더
        adapter_name=row["이름"],  # 어댑터를 켤 때 부를 이름
    )

### 3-3. 에폭별 어댑터 평가

어댑터를 하나씩 켜서 3-1과 같은 지표 일곱 개를 계산하고, 지표 테이블의 4비트 Base 행 아래에 어댑터마다 행을 더합니다. 판사 호출은 119번 × 어댑터 4개입니다.

In [ ]:
# 어댑터를 하나씩 켜서 채점 테이블과 지표 일곱 개를 계산합니다(판사 호출마다 나오는 DeepEval 요약과 경고는 3-1과 같으며 오류가 아닙니다)
epoch_tables = {}
for row in checkpoint_rows:
    qlora_model.set_adapter(
        row["이름"],          # 켤 어댑터 이름
        inference_mode=True,  # 켠 어댑터를 학습하지 않게 고정합니다
    )
    epoch_tables[row["이름"]] = evaluate_model(qlora_model)
    model_metrics[row["이름"]] = table_metrics(epoch_tables[row["이름"]])

metrics_table = pd.DataFrame(model_metrics).T.rename(columns=METRIC_NAMES)
display(metrics_table.round(3))

**결과 읽기**
- **학습 데이터와 재현:** 강사 실행에서 학습 데이터 정답률은 4비트 Base 0.000에서 ep1 0.225, ep2 0.350, ep3 0.350, ep4 0.475로 오릅니다. 재현 정답률은 ep1~ep4 모두 0.071(28문항 가운데 2개)로, 에폭이 늘어도 오르지 않습니다.
- **적용:** ep1만 21문항 가운데 1개를 맞힙니다.
- **일반 능력:** 일반 질문 정답 수는 ep1~ep4가 7, 7, 6, 5개로 줄어듭니다. 일반 perplexity는 4비트 Base 50.1에서 ep1~ep4가 58.3, 68.2, 74.7, 77.6으로 오릅니다.
- **읽는 법:** 에폭이 늘수록 학습 데이터 정답률만 오르고, 재현과 적용은 그대로이며, 일반 능력은 떨어집니다. 어댑터가 학습 데이터의 답을 외운다는 뜻입니다.

### 3-4. QLoRA Tuned 고르기

에폭별 어댑터 4개 가운데 4절 비교에 쓸 하나를 지난 시간 교안 02와 같은 규칙으로 고릅니다. 고른 어댑터를 이 교안에서는 QLoRA Tuned라고 부릅니다.
1. **학습 데이터 정답률 0.8 이상만 후보로 둡니다.** 학습이 제대로 됐다면 이미 본 답은 대부분 맞혀야 합니다.
2. **후보 가운데 노무 평가셋 통과 수(재현 + 적용)가 가장 많은 것을 고릅니다.** 같으면 앞 에폭을 고릅니다.
3. **후보가 없으면 학습 데이터 정답률이 가장 높은 것을 고릅니다.** 같으면 앞 에폭을 고릅니다.

In [ ]:
TRAIN_ACC_MIN = 0.8   # 규칙 1: 학습 데이터 정답률이 이 값 이상인 어댑터만 후보

# 규칙 2: 후보 가운데 노무 평가셋 통과 수가 가장 많은 것(앞 에폭부터 보므로 같으면 앞 에폭)
tuned_row = None
for row in checkpoint_rows:
    passes = epoch_tables[row["이름"]]["domain"]["통과"].sum()   # 재현 + 적용 통과 문항 수
    if model_metrics[row["이름"]]["train_acc"] >= TRAIN_ACC_MIN:
        if tuned_row is None or passes > best_passes:
            tuned_row = row
            best_passes = passes

# 규칙 3: 후보가 없으면 학습 데이터 정답률이 가장 높은 것(같으면 앞 에폭)
if tuned_row is None:
    tuned_row = checkpoint_rows[0]
    for row in checkpoint_rows:
        if model_metrics[row["이름"]]["train_acc"] > model_metrics[tuned_row["이름"]]["train_acc"]:
            tuned_row = row

# 고른 어댑터의 이름과, 테이블에 쓸 이름표(지난 시간 LoRA Tuned 이름표와 같은 모양)
tuned_name = tuned_row["이름"]
tuned_label = f"QLoRA Tuned {tuned_row['에폭']}에폭(step {tuned_row['갱신 횟수']})"
tuned_passes = epoch_tables[tuned_name]["domain"]["통과"].sum()
print("고른 어댑터:", tuned_label, "| 학습 데이터 정답률:", model_metrics[tuned_name]["train_acc"],
      "| 노무 평가셋 통과 수:", tuned_passes)

**결과 읽기**
- **고른 어댑터:** 강사 실행에서는 학습 데이터 정답률이 0.8 이상인 어댑터가 없었습니다. 그래서 규칙 3으로 학습 데이터 정답률이 가장 높은 ep4(0.475)를 골랐습니다. 지난 시간 LoRA도 같은 규칙으로 ep4를 골랐습니다.
- **차이 읽기:** 판사의 판정이 통과와 실패의 경계에 있는 문항은 실행마다 결과가 달라질 수 있습니다. 그래서 모델 사이의 한두 문항 차이는 차이 없음으로 읽습니다.

## 4. LoRA·풀 파인튜닝과 비교

QLoRA 논문은 4비트 베이스 위에서 학습한 어댑터가 bf16 LoRA와 비슷한 성능을 낸다고 보고합니다([Dettmers et al., 2023](https://arxiv.org/abs/2305.14314)). 이 노무 데이터에서도 그런지 QLoRA Tuned를 지난 시간 LoRA Tuned, SFT 단원 풀 파인튜닝 Tuned와 한 테이블에서 비교합니다. 학습률, 갱신 횟수, 어댑터를 고르는 방법이 같으므로, QLoRA Tuned와 LoRA Tuned의 차이는 베이스 정밀도(4비트와 float32)와 어댑터 dtype(bf16과 fp32)에서 옵니다.

- **비교하는 모델:** 4비트 Base와 QLoRA Tuned(이 교안), float32 Base와 LoRA Tuned(지난 시간), 풀 파인튜닝 Tuned(SFT 단원)입니다.
- **학습 시간:** 지난 시간 LoRA Tuned의 학습 시간은 `day71_PEFT_LoRA/output/mlflow.db`(실험 `labor-lora`, run `lora_lr_2e-4`)에 남은 `train_runtime`(초)입니다.
- **학습 메모리:** 교안 01 6-2에서 같은 조건으로 측정한 학습 메모리 값을 씁니다.
- **옮겨 적는 값:** 아래 셀에 숫자로 적힌 지난 시간 값은 강사 실행 값이고, `None`은 아직 옮기지 않은 값입니다. 자기 노트북의 값이 있으면 그 값으로 바꿔 넣습니다.

In [ ]:
# 지난 시간 교안 02 full_compare_table 의 Base 행. 적힌 값은 그 테이블의 "Base(지난 시간)" 행과 같은 SFT 단원 교안 02의 강사 실행 값입니다
float32_base_metrics = {"train_acc": 0.025, "domain_recall_acc": 0.036, "domain_apply_acc": 0.0, "over_refusal_rate": 0.0, "refusal_rate": 0.0, "general_acc": 0.033, "general_ppl": 36.43}

# 지난 시간 교안 02 full_compare_table 의 "LoRA Tuned <에폭>에폭(step <갱신 횟수>)" 행. 자기 값을 옮겨 적습니다(None 은 아직 옮기지 않은 값)
lora_tuned_metrics = {"train_acc": None, "domain_recall_acc": None, "domain_apply_acc": None, "over_refusal_rate": None, "refusal_rate": None, "general_acc": None, "general_ppl": None}

# SFT 단원 교안 02 지표 테이블의 "Tuned 3에폭(step 879)" 행(강사 실행 값. 자기 결과가 있으면 바꿔 넣습니다)
full_ft_metrics = {"train_acc": 0.85, "domain_recall_acc": 0.286, "domain_apply_acc": 0.048, "over_refusal_rate": 0.0, "refusal_rate": 0.333, "general_acc": 0.167, "general_ppl": 59.71}

# 학습 시간(옮겨 적는 값). MLflow run 에 남은 train_runtime 입니다
lora_train_seconds = None     # 옮겨 적는 값: day71_PEFT_LoRA/output/mlflow.db 의 실험 "labor-lora", run "lora_lr_2e-4" 의 train_runtime(초)
full_ft_train_seconds = None  # 옮겨 적는 값: SFT 단원 풀 파인튜닝 학습 run 의 train_runtime(초)

# 다섯 모델의 지표 일곱 개를 한 테이블로 둡니다
compare_table = pd.DataFrame({
    "4비트 Base": model_metrics["4비트 Base"],
    "float32 Base(지난 시간)": float32_base_metrics,
    tuned_label: model_metrics[tuned_name],
    "LoRA Tuned(지난 시간)": lora_tuned_metrics,
    "풀 파인튜닝 Tuned(SFT 단원)": full_ft_metrics,
}).T.rename(columns=METRIC_NAMES)

# 학습 시간 열을 더합니다. Base 두 행은 학습하지 않았으므로 None 입니다
compare_table["학습 시간(초)"] = [None, None, train_seconds, lora_train_seconds, full_ft_train_seconds]

# None 칸이 NaN 으로 보이도록 숫자로 바꿔 반올림합니다
display(compare_table.astype(float).round(3))

**결과 읽기**
- **4비트로 불러온 영향:** 강사 실행에서 4비트 Base의 일반 perplexity는 50.1로 float32 Base(36.4)보다 약 37% 높습니다. 두 Base의 판사 지표는 한 문항씩만 달라 차이 없음으로 봅니다.
- **QLoRA Tuned와 LoRA Tuned:** 강사 실행에서 QLoRA Tuned는 LoRA Tuned보다 학습 데이터 정답률(0.475와 0.625), 재현, 적용, 일반 질문이 모두 조금씩 낮습니다. 일반 perplexity는 더 높습니다(77.6과 61.5).
- **풀 파인튜닝 Tuned와 비교:** 풀 파인튜닝 Tuned는 학습 데이터 정답률 0.850, 재현 28문항 가운데 8개로 도메인을 가장 많이 배웠습니다. QLoRA Tuned는 학습 데이터 정답률 0.475, 재현 2개로 덜 배웠습니다. 일반 perplexity도 QLoRA Tuned(77.6)가 풀 파인튜닝 Tuned(59.7)보다 높으므로, QLoRA Tuned가 일반 능력을 덜 잃었다고 볼 수도 없습니다.

**LoRA와 QLoRA 고르기**
- **결정 규칙:** 베이스를 bf16으로 불러와 학습해도 학습 메모리가 GPU 메모리에 들어가면 LoRA를, 들어가지 않으면 QLoRA를 씁니다. 교안 01의 9B 모델(24GB GPU)은 QLoRA가 필요한 경우이고, 이 교안의 0.8B 모델은 LoRA로도 GPU 메모리에 들어갑니다.
- **Qwen3.5 주의:** [Unsloth 미세조정 안내](https://unsloth.ai/docs/models/qwen3.5/fine-tune)는 Qwen3.5에 4비트 QLoRA를 권하지 않습니다. Qwen3.5는 양자화 오차가 크기 때문입니다. 그래서 bf16 LoRA가 GPU 메모리에 들어가면 bf16 LoRA를 먼저 씁니다.
- **실무 흐름:** 작은 GPU에서 QLoRA로 데이터와 설정을 빠르게 실험하고, 최종 학습은 큰 GPU에서 bf16 LoRA로 하는 경우가 많습니다. 어느 쪽이든 결과는 같은 모양의 어댑터 파일이라 저장, 병합, 서빙 방법이 같습니다.

## 5. 어댑터 저장과 병합

### 5-1. 저장된 어댑터

`SFTTrainer`가 에폭마다 체크포인트 폴더에 어댑터만 저장했으므로, QLoRA Tuned의 폴더가 곧 쓸 어댑터입니다. 4비트 Base는 저장하지 않습니다.

- **어댑터 크기 계산:** 5,411,328개 × 2바이트(bf16) / 1024² ≈ 10.32MB입니다.
- **지난 시간 LoRA:** 같은 개수 × 4바이트(float32)라 약 20.64MB입니다.

### 5-2. 원래 정밀도 베이스에 병합

병합 방법(`merge_and_unload`)은 지난 시간과 같고, QLoRA 어댑터에서는 병합할 베이스가 다릅니다.

- **4비트 베이스에 바로 병합하지 않습니다.** 4비트 가중치를 복원해 어댑터를 더한 뒤 다시 4비트 단계 값으로 맞추므로 양자화 오차가 한 번 더 생깁니다([PEFT 양자화 문서](https://huggingface.co/docs/peft/main/en/developer_guides/quantization)). 그래서 이 교안은 베이스를 원래 정밀도(float32)로 새로 불러와 어댑터를 얹고 병합합니다.
- **배포 전에는 병합 모델도 다시 평가합니다.** 어댑터는 4비트 단계 값으로 바뀐 가중치에 맞춰 학습했는데 병합 베이스는 4비트로 바꾸기 전 float32 가중치라, 병합 모델의 답이 4비트 + 어댑터의 답과 달라질 수 있기 때문입니다.

<img src="images/16_merge_full_precision.png" width="1050" alt="QLoRA 어댑터는 원래 정밀도(float32)로 새로 불러온 베이스에 병합합니다. 4비트 베이스에 병합하면 다시 4비트 단계로 맞추면서 양자화 오차가 한 번 더 생기므로, 병합 모델은 다시 평가합니다.">

In [ ]:
# 원래 정밀도(float32)로 베이스를 새로 불러옵니다
merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,           # 지난 시간과 같은 모델
    dtype=torch.float32,  # 원래 정밀도
).to(device)

# QLoRA Tuned 어댑터 하나만 얹고 베이스 가중치에 더해 모델 하나로 만듭니다
merged_model = PeftModel.from_pretrained(
    merge_base,          # float32 베이스
    tuned_row["폴더"],   # QLoRA Tuned 체크포인트 폴더
).merge_and_unload()

In [ ]:
# 병합 모델에 노무 평가셋 첫 질문을 묻습니다(어댑터 없이 모델 하나로 답합니다)
merged_question = eval_rows[0]["question"]
print("질문:", merged_question)
print("병합 모델 답변:", generate_reply(merged_model, merged_question))

평가와 병합에 쓴 모델을 지워 함께 따라하기에 쓸 GPU 메모리를 비웁니다.

In [ ]:
# 평가와 병합에 쓴 모델을 지웁니다(eval_base 는 qlora_model 안에 들어 있어 함께 지웁니다. 한 번만 실행)
del merged_model, merge_base, qlora_model, eval_base
gc.collect()
torch.cuda.empty_cache()

## 6. 🖐️ 함께 따라하기: 전월세 상담 QLoRA 어댑터 학습과 병합

본실습의 QLoRA 학습을 지난 시간 함께 따라하기의 전월세 상담 데이터에 적용합니다. 지난 시간 전월세 LoRA와 같은 데이터, 같은 LoRA 설정에 베이스만 4비트로 바꿔 학습하고, 본실습 2-3처럼 학습 시간과 최대 할당 메모리를 직접 기록합니다. 마지막에 본실습 5-2처럼 어댑터를 원래 정밀도 베이스에 병합해 답을 얻습니다.

**본실습과 다른 점**
- **데이터:** `data/따라하기/lease_train.jsonl`, `lease_val.jsonl`. 지난 시간 함께 따라하기와 같은 주택임대차보호법 질문-답변입니다.
- **시스템 프롬프트:** `SYSTEM_PROMPT_LEASE`. 학습 데이터의 시스템 메시지와 같은 문장이고, 답을 만들 때도 이 문장을 넘깁니다.
- **갱신 횟수:** 학습 행이 137개라 에폭마다 35번, 모두 140번 갱신하므로 `logging_steps`는 10으로 줄입니다. 체크포인트는 `output/lease_qlora_lr2e-4`에 에폭마다 저장합니다.
- **기록:** run 이름은 `lease_qlora_lr_2e-4`이고, 본실습 run과 섞이지 않게 MLflow 실험 `lease-qlora`에 남깁니다.

**실습 진행 방법:** 작성 셀의 `...`만 주석에 맞춰 채웁니다. 빈칸은 6-4 학습 셀 다음 셀의 학습 시간과 최대 할당 메모리 두 줄, 6-6의 병합 한 줄에만 있고, 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

**확인 기준**
- 학습 파라미터 수가 본실습과 같은 5,411,328개입니다.
- 학습 시간(초)과 최대 할당 메모리(GB)가 출력되고, 같은 값이 MLflow run에 남습니다.
- `output/lease_qlora_lr2e-4`에 체크포인트 폴더가 4개(35, 70, 105, 140) 생깁니다.
- 6-6의 병합 모델이 검증 데이터 질문에 답을 냅니다.

### 6-1. 데이터 읽기

In [ ]:
# 전월세 데이터의 시스템 메시지와 글자까지 같은 문장
SYSTEM_PROMPT_LEASE = "너는 전월세 상담 챗봇이다. 주택임대차보호법 조문에 근거해 존댓말로 간결히 답한다."

# 지난 시간 따라하기에서 나눠 저장한 전월세 학습, 검증 파일을 읽고 Dataset으로 만듭니다
lease_train_rows = load_jsonl(data_dir / "따라하기" / "lease_train.jsonl")
lease_val_rows = load_jsonl(data_dir / "따라하기" / "lease_val.jsonl")
lease_train_dataset = Dataset.from_list([to_prompt_completion(row) for row in lease_train_rows])
lease_val_dataset = Dataset.from_list([to_prompt_completion(row) for row in lease_val_rows])
print("학습 행 수:", len(lease_train_dataset), "| 검증 행 수:", len(lease_val_dataset))

### 6-2. 4비트로 불러오기와 학습 전 답

In [ ]:
# 4비트 Base를 새로 불러옵니다(본실습 어댑터와 따로 학습합니다)
lease_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,                            # 본실습과 같은 모델
    quantization_config=bnb_config,        # 본실습과 같은 4비트 설정
    dtype=torch.bfloat16,                  # 4비트로 바뀌지 않는 층의 dtype
    device_map={"": 0},                    # 모델 전체("")를 GPU 0번(첫 번째 GPU)에 둡니다
)

In [ ]:
# 학습한 질문 하나(학습 데이터 첫 행)와 학습에 없는 질문 하나(검증 데이터 첫 행)를 4비트 Base에 묻습니다
lease_questions = [lease_train_rows[0]["question"], lease_val_rows[0]["question"]]
lease_before = []
for question in lease_questions:
    lease_before.append(generate_reply(
        lease_base,           # 학습 전 4비트 Base
        question,             # 묻는 질문
        SYSTEM_PROMPT_LEASE,  # 전월세 데이터와 같은 시스템 프롬프트
    ))
lease_table = pd.DataFrame({
    "질문": lease_questions,
    "정답 답변": [lease_train_rows[0]["answer"], lease_val_rows[0]["answer"]],
    "4비트 Base 답변": lease_before,
})
display(lease_table)

### 6-3. 학습 설정과 트레이너

In [ ]:
# 본실습과 같은 설정에 폴더, 기록 간격, run 이름만 바꿉니다
lease_config = SFTConfig(
    output_dir=str(output_dir / "lease_qlora_lr2e-4"),  # 에폭마다 checkpoint-<갱신 횟수> 폴더가 생깁니다
    learning_rate=2e-4,                        # 본실습과 같은 학습률
    num_train_epochs=4,                          # 본실습과 같은 에폭 수
    per_device_train_batch_size=4,               # 지난 시간 전월세 LoRA와 같은 배치
    seed=0,                                      # 본실습과 같은 시드
    gradient_checkpointing=False,                # 본실습과 같이 끕니다
    max_length=1024,                             # 본실습과 같은 길이 상한
    eval_strategy="epoch",                       # 에폭마다 검증 손실을 계산합니다
    save_strategy="epoch",                       # 에폭마다 어댑터를 저장합니다
    save_only_model=True,                        # AdamW 상태 없이 어댑터만 저장합니다
    logging_steps=10,                            # 갱신 횟수가 적어 기록 간격을 줄입니다
    eval_on_start=True,                          # 학습 전 검증 손실도 기록합니다
    report_to=["mlflow"],                        # 기록을 MLflow에 남깁니다
    run_name="lease_qlora_lr_2e-4",               # MLflow run 이름
)

# 본실습 2-3처럼 4비트 Base 와 본실습의 lora_config 를 함께 넘깁니다. 어댑터는 트레이너가 붙입니다
lease_trainer = SFTTrainer(
    model=lease_base,                  # 어댑터를 붙이지 않은 4비트 Base
    args=lease_config,                 # 위 학습 설정
    train_dataset=lease_train_dataset, # 전월세 학습 데이터
    eval_dataset=lease_val_dataset,    # 전월세 검증 데이터
    processing_class=tokenizer,        # 본실습과 같은 토크나이저
    peft_config=lora_config,           # 본실습과 같은 LoRA 설정
)

# 트레이너가 어댑터를 붙인 모델(PeftModel)
lease_model = lease_trainer.model
lease_model.print_trainable_parameters()

### 6-4. 학습

In [ ]:
# 전월세 run 은 따로 둔 실험에 남깁니다. 실험을 새로 만든다는 INFO 줄은 오류가 아닙니다
mlflow.set_experiment("lease-qlora")

# 학습하는 동안의 최대 할당 메모리만 남기려고 지금까지의 최댓값 기록을 지웁니다(본실습 2-3과 같은 방법)
torch.cuda.reset_peak_memory_stats()

# 데이터 파일과 행 수를 run에 남기고 학습합니다(본실습과 같은 방법)
with mlflow.start_run(run_name=lease_config.run_name) as lease_run:
    mlflow.log_params({
        "train_file": "lease_train.jsonl",       # 학습 파일
        "train_rows": len(lease_train_dataset),  # 학습 행 수
        "val_file": "lease_val.jsonl",           # 검증 파일
        "val_rows": len(lease_val_dataset),      # 검증 행 수
        "load_in_4bit": True,                    # 베이스를 4비트로 불러왔습니다
    })
    # 본실습 2-3과 같은 PAD/BOS/EOS 토큰 안내가 나옵니다. 오류가 아닙니다
    lease_output = lease_trainer.train()

In [ ]:
# 🖐️ 함께 따라하기: 학습 시간(초)과 학습하는 동안의 최대 할당 메모리(GB)를 본실습 2-3처럼 꺼내세요
# 최대 할당 메모리 기록은 다음 reset_peak_memory_stats 전까지 남아 있어 학습이 끝난 뒤 꺼내도 같은 값입니다
lease_train_seconds = ...     # 학습 결과(lease_output)의 metrics 에 든 train_runtime
lease_max_allocated_gb = ...  # torch.cuda 가 기록한 최대 할당 메모리(바이트)를 GB로

# 끝난 전월세 run 을 run_id 로 다시 열어 최대 할당 메모리를 남깁니다
with mlflow.start_run(run_id=lease_run.info.run_id):
    mlflow.log_metric("max_memory_allocated_gb", lease_max_allocated_gb)
print("학습 시간(초):", round(lease_train_seconds), "| 최대 할당 메모리(GB):", round(lease_max_allocated_gb, 2))

### 6-5. 학습 전·후 답과 체크포인트

In [ ]:
# 드롭아웃을 끄고 같은 두 질문을 학습한 모델에 물어 열로 더합니다
lease_model.eval()
lease_after = []
for question in lease_questions:
    lease_after.append(generate_reply(
        lease_model,          # 학습한 QLoRA 모델
        question,             # 묻는 질문
        SYSTEM_PROMPT_LEASE,  # 전월세 데이터와 같은 시스템 프롬프트
    ))
lease_table["QLoRA 답변"] = lease_after
display(lease_table)

In [ ]:
# 에폭마다 저장한 체크포인트 폴더의 갱신 횟수
lease_steps = []
for path in (output_dir / "lease_qlora_lr2e-4").glob("checkpoint-*"):
    lease_steps.append(int(path.name.split("-")[1]))
lease_steps.sort()
print("체크포인트 갱신 횟수:", lease_steps)

### 6-6. 원래 정밀도 베이스에 병합하고 답 얻기

함께 따라하기는 에폭을 고르지 않았으므로 마지막 에폭 어댑터를 병합합니다.

In [ ]:
# 본실습 5-2처럼 원래 정밀도(float32)로 베이스를 새로 불러옵니다
lease_merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,           # 본실습과 같은 모델
    dtype=torch.float32,  # 원래 정밀도
).to(device)

# 마지막 에폭 체크포인트 폴더
lease_adapter_dir = output_dir / "lease_qlora_lr2e-4" / f"checkpoint-{lease_steps[-1]}"

# 🖐️ 함께 따라하기: lease_adapter_dir 의 어댑터를 lease_merge_base 에 얹고 병합하세요(본실습 5-2와 같은 방법)
lease_merged = ...

In [ ]:
# 병합 모델에 검증 데이터 질문(학습에 없는 질문)을 묻습니다
print("질문:", lease_questions[1])
print("병합 모델 답변:", generate_reply(lease_merged, lease_questions[1], SYSTEM_PROMPT_LEASE))

## 7. 핵심 정리

- **QLoRA 학습:** 4비트 Base와 지난 시간과 같은 `lora_config`, `SFTConfig`를 `SFTTrainer`에 넘기면, 트레이너가 어댑터를 붙이고 bf16으로 바꿔 학습합니다.
- **QLoRA Tuned 고르기:** 에폭 어댑터 4개를 지난 시간과 같은 판사 지표와 규칙으로 평가해 하나를 고릅니다(강사 실행에서 ep4).
- **LoRA·풀 파인튜닝과 비교:** 강사 실행에서 QLoRA Tuned는 LoRA Tuned보다 조금 덜 배웠습니다. 4비트로 불러오기만 해도 Base의 일반 perplexity가 약 37% 올랐습니다(float32 Base 36.4 -> 4비트 Base 50.1).
- **LoRA와 QLoRA 고르기:** 베이스를 bf16으로 불러와 학습해도 GPU 메모리에 들어가면 LoRA를, 들어가지 않으면 QLoRA를 씁니다.
- **저장과 병합:** 어댑터만 bf16으로 저장됩니다(약 10.32MB). 병합은 원래 정밀도(float32) 베이스에 합니다. 병합 모델의 답은 4비트 + 어댑터의 답과 달라질 수 있으므로 배포 전에 다시 평가합니다.